# Training cost vs. test log-likelihood (Pareto front)

Camera-ready cost analysis (meta-review: total training cost of ALC vs. DE). Cost is measured in optimizer steps, which is device-independent (wall-clock times are in the paper's runtime table).

* **step budget**: steps the run actually trains for (no early stopping in our code).
* **steps to best val. checkpoint**: steps until the checkpoint selected by validation log-likelihood; per anchor for DE, anchors + curve phase for ALC.

Inputs (produced by the scripts in `rebuttal_stacking/q1_compute/` from W&B histories, and by the table cells of `analyse_qwen_only_new DS.ipynb`):
`rebuttal_stacking/result/df_plot_table.csv`, `rebuttal_stacking/q1_compute/steps_to_best.csv`, `rebuttal_stacking/q1_compute/fge_warmup_best.csv`.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["JAX_PLATFORMS"] = "cpu"

import wandb
from tqdm import tqdm
from copy import deepcopy
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

n_name = r"$N$"
free_model_name = "FLC"
pretrained_model_name = "ALC"

%load_ext autoreload
%autoreload 2

In [ ]:
def get_figsize(columnwidth, wf=0.5, hf=(5.**0.5-1.0)/2.0, ):
    """Parameters:
    - wf [float]:  width fraction in columnwidth units
    - hf [float]:  height fraction in columnwidth units.
                    Set by default to golden ratio.
    - columnwidth [float]: width of the column in latex. Get this from LaTeX
                            using \\showthe\\columnwidth
    Returns:  [fig_width,fig_height]: that should be given to matplotlib
    """
    fig_width_pt = columnwidth*wf
    inches_per_pt = 1.0/72.27               # Convert pt to inch
    fig_width = fig_width_pt*inches_per_pt  # width in inches
    fig_height = fig_width*hf      # height in inches
    return [fig_width, fig_height]


fig_width_pt = 397.48499  # Get this from LaTeX using \showthe\columnwidth
inches_per_pt = 1.0/72.27               # Convert pt to inches
golden_mean = (np.sqrt(5)-1.0)/2.0         # Aesthetic ratio
fig_width = fig_width_pt*inches_per_pt  # width in inches
fig_height = fig_width*golden_mean       # height in inches
figsize = [fig_width, fig_height]
print("Figure size=", figsize)
tex_fonts = {
    "text.usetex": True,
    "font.family": "Nimbus Sans",
    "axes.labelsize": 10,
    "font.size": 10,
    "legend.fontsize": 9,
    "legend.title_fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.figsize": figsize
}
plt.rcParams.update(tex_fonts)

In [ ]:
import re

ll_table = pd.read_csv("rebuttal_stacking/result/df_plot_table.csv")
ll_mean = ll_table[ll_table["Metric"] == "ll"].groupby(["Dataset", "Model"])["value"].mean()
steps = pd.read_csv("rebuttal_stacking/q1_compute/steps_to_best.csv")
fge = pd.read_csv("rebuttal_stacking/q1_compute/fge_warmup_best.csv")
fge = fge[fge["n"] == 17]

SWA_STEPS = 2000
FGE_STEPS = 2000
FGE_WARMUP = 5000
DATASETS = ["winogrande_s", "winogrande_m", "obqa", "ARC-Easy", "ARC-Challenge", "boolq"]
N_LIST = [2, 3, 5, 9]

rows = []
for ds in DATASETS:
    alc = steps[(steps["dataset"] == ds) & (steps["method"] == "ALC")]
    P, S = alc["P"].median(), alc["S"].median()

    def add(model, family, n, budget, best):
        rows.append(dict(Dataset=ds, Model=model, Family=family, N=n, budget=budget, best=best,
                         ll=ll_mean.get((ds, model), np.nan)))

    add("MAP", "MAP", 1, P, alc["map_best"].median())
    add("SWA", "SWA", 1, SWA_STEPS, SWA_STEPS)
    f = fge[fge["dataset"] == ds]
    if len(f):
        add("FGE(17)", "FGE", 17, FGE_WARMUP + FGE_STEPS, f["warm_best"].median() + FGE_STEPS)
    for n in N_LIST:
        a = alc[alc["N"] == n]
        add(f"DE({n})", "DE", n, n * P, a["de_best"].median())
        add(f"{pretrained_model_name}({n},2)", pretrained_model_name, n, n * P + S, a["best"].median())
        fl = steps[(steps["dataset"] == ds) & (steps["method"] == "FLC") & (steps["N"] == n)]
        if len(fl):
            add(f"{free_model_name}({n},2)", free_model_name, n, fl["budget"].median(), fl["best"].median())

df_pareto = pd.DataFrame(rows).dropna(subset=["ll"])
df_pareto.to_csv("result/pareto_points.csv", index=False)
df_pareto.head(15)

In [ ]:
def pareto_front(d, x):
    front, best = [], -np.inf
    for _, r in d.sort_values([x, "ll"], ascending=[True, False]).iterrows():
        if r["ll"] > best:
            front.append(r)
            best = r["ll"]
    return pd.DataFrame(front)


for x in ["budget", "best"]:
    print(f"--- {x}")
    for ds in DATASETS:
        print(f"{ds:14s}", list(pareto_front(df_pareto[df_pareto["Dataset"] == ds], x)["Model"]))

In [ ]:
_pal = sns.color_palette("tab20c")
family_order = ["MAP", "SWA", "FGE", "DE", pretrained_model_name, free_model_name]
family_palette = {
    "MAP": "black",
    "SWA": _pal[17],
    "FGE": _pal[12],
    "DE": _pal[5],
    pretrained_model_name: _pal[0],
    free_model_name: _pal[8],
}
family_marker = {"MAP": "s", "SWA": "X", "FGE": "P", "DE": "o", pretrained_model_name: "o", free_model_name: "o"}
x_labels = {"budget": r"step budget [$10^3$ steps]", "best": r"steps to best val.\ checkpoint [$10^3$ steps]"}

fig, axs = plt.subplots(2, len(DATASETS), figsize=(fig_width * 1.35, fig_width * 0.55), sharey="col")
for row, x in enumerate(["budget", "best"]):
    for col, ds in enumerate(DATASETS):
        ax = axs[row, col]
        d = df_pareto[df_pareto["Dataset"] == ds]
        fr = pareto_front(d, x)
        xs = np.r_[fr[x].values, d[x].max() * 1.3] / 1000
        ax.step(xs, np.r_[fr["ll"].values, fr["ll"].values[-1]], where="post",
                color="gray", ls="--", lw=0.8, alpha=0.7, zorder=1)
        for fam in family_order:
            s = d[d["Family"] == fam].sort_values("N")
            if s.empty:
                continue
            if len(s) > 1:
                ax.plot(s[x] / 1000, s["ll"], color=family_palette[fam], lw=0.8, alpha=0.6, zorder=2)
            ax.scatter(s[x] / 1000, s["ll"], color=family_palette[fam], marker=family_marker[fam],
                       s=10 + 4 * np.sqrt(s["N"]), edgecolor="white", linewidth=0.3, zorder=3, label=fam)
        ax.set_xscale("log")
        ax.grid(True, axis="x", linestyle="--", alpha=0.4)
        ax.tick_params(labelsize=7)
        if row == 0:
            ax.set_title(ds.replace("_", r"\_"), fontsize=9)
        if col == 0:
            ax.set_ylabel(r"Log-Likelihood $\uparrow$", fontsize=8)
        ax.set_xlabel(x_labels[x] if col == len(DATASETS) // 2 else "", fontsize=8)
        ax.xaxis.set_label_coords(0.0, -0.22) if col == len(DATASETS) // 2 else None

handles, labels = axs[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.04), ncol=len(family_order), frameon=False, fontsize=8)
fig.tight_layout()
fig.subplots_adjust(wspace=0.35, hspace=0.45)

os.makedirs("result", exist_ok=True)
plt.savefig("result/pareto_steps.pdf", dpi=300, bbox_inches="tight")
plt.savefig("result/pareto_steps.png", dpi=150, bbox_inches="tight")
plt.show()